# Regresión Lineal en Hospitalizaciones (GRD)

## Curso: Análisis de Datos e Inferencia Estadística

Este notebook introduce los modelos de regresión lineal usando ejemplos realistas de hospitalizaciones.

Incluye:
- Explicación conceptual
- Ejemplos en salud
- Interpretación de modelos
- Aplicación a GRD


# 1. ¿Qué es un modelo de regresión?

Un modelo de regresión es una herramienta que permite:

👉 **explicar o predecir una variable a partir de otras variables**

---

## Ejemplo en hospitalizaciones

Queremos entender:

👉 ¿Qué factores explican el costo hospitalario?

Variables posibles:
- Edad
- Días de estadía
- Severidad
- Servicio clínico
- Tipo de egreso
- GRD

El modelo aprende una relación como:

Costo = f(edad, días, severidad, servicio, etc.)

---

## ¿Para qué sirve?

- Predecir costos hospitalarios  
- Entender qué factores aumentan la estadía  
- Comparar servicios clínicos  
- Analizar uso de recursos  

---

## Tipos de regresión

- Regresión lineal → variables continuas (costo, días)
- Regresión logística → variables binarias (fallecido vs alta)

---

## Idea clave

👉 El modelo NO descubre causalidad  
👉 Solo identifica asociaciones en los datos


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

# 2. Simulación de datos hospitalarios (GRD)

En esta sección generamos un conjunto de datos simulados que representa hospitalizaciones, inspirado en la lógica de los sistemas GRD (Grupos Relacionados por el Diagnóstico).

El objetivo no es trabajar con datos reales, sino contar con un dataset controlado que nos permita entender cómo funcionan los modelos de regresión.

Cada fila del dataset representa un paciente hospitalizado, e incluye variables clínicas y administrativas relevantes, como:

* Edad del paciente
* Sexo
* Servicio clínico (Medicina, Cirugía, UCI)
* Tipo de egreso (Alta, Fallecido, Traslado)
* Nivel de severidad
* Grupo GRD
* Número de diagnósticos y procedimientos
* Días de estadía hospitalaria
* Costo hospitalario

Las variables outcome (días de estadía y costo) se generan de forma que dependan de factores clínicos reales. Por ejemplo:

* Pacientes más graves tienden a permanecer más días hospitalizados
La UCI se asocia a mayores costos
* Más procedimientos implican mayor uso de recursos

👉 Esto permite que el modelo de regresión recupere relaciones que tienen sentido clínico, facilitando su interpretación.

In [2]:
n = 400

edad = np.random.normal(60, 18, n).clip(18, 95)
sexo = np.random.choice(["F", "M"], n)
servicio = np.random.choice(["Medicina", "Cirugía", "UCI"], n, p=[0.5, 0.3, 0.2])
egreso = np.random.choice(["Alta", "Fallecido", "Traslado"], n, p=[0.85, 0.1, 0.05])
severidad = np.random.choice(["Baja", "Media", "Alta"], n, p=[0.5, 0.35, 0.15])
grd = np.random.choice(["GRD1", "GRD2", "GRD3"], n)

num_diag = np.random.poisson(3, n)+1
num_proc = np.random.poisson(2, n)

dias = (2 + 0.05*edad + 0.8*num_diag + 0.6*num_proc +
        np.where(servicio=="UCI", 5, 0) +
        np.where(severidad=="Alta", 4, np.where(severidad=="Media",2,0)) +
        np.random.normal(0,2,n)).clip(1)

costo = (300000 +
         8000*edad +
         150000*dias +
         100000*num_diag +
         200000*num_proc +
         np.where(servicio=="UCI", 1500000, 0) +
         np.where(severidad=="Alta", 1200000, np.where(severidad=="Media",500000,0)) +
         np.random.normal(0,800000,n)).clip(200000)

df = pd.DataFrame({
    "edad": edad,
    "sexo": sexo,
    "servicio": servicio,
    "egreso": egreso,
    "severidad": severidad,
    "grd": grd,
    "num_diag": num_diag,
    "num_proc": num_proc,
    "dias_estadia": dias,
    "costo": costo
})

df.head()

,edad,sexo,servicio,egreso,severidad,grd,num_diag,num_proc,dias_estadia,costo
0,68.940855,F,Medicina,Alta,Baja,GRD3,2,1,9.567652,2.210247e+06
1,57.511243,F,Cirugía,Alta,Baja,GRD3,1,2,7.800529,3.150596e+06
2,71.658394,F,Medicina,Alta,Media,GRD1,5,2,10.424181,5.196443e+06
3,87.414537,F,Medicina,Alta,Media,GRD2,4,2,13.115236,3.945166e+06
4,55.785239,M,Medicina,Alta,Baja,GRD3,5,1,10.732324,3.601220e+06


# 3. Modelo de regresión lineal múltiple

In [3]:
modelo = smf.ols(
    "costo ~ edad + dias_estadia + num_diag + num_proc + C(sexo) + C(servicio) + C(egreso) + C(severidad) + C(grd)",
    data=df
).fit()

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  costo   R-squared:                       0.722
Model:                            OLS   Adj. R-squared:                  0.713
Method:                 Least Squares   F-statistic:                     77.23
Date:                Mon, 27 Apr 2026   Prob (F-statistic):           9.71e-99
Time:                        20:33:33   Log-Likelihood:                -5987.4
No. Observations:                 400   AIC:                         1.200e+04
Df Residuals:                     386   BIC:                         1.206e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                1

# 4. Interpretación del modelo

## Coeficientes
Cada coeficiente indica cuánto cambia el costo cuando cambia esa variable, manteniendo las demás constantes.

## Intercepto
Costo base cuando todas las variables están en 0 o categoría referencia.

## Variables categóricas
Se interpretan respecto a una categoría base.

## F-statistic
Indica si el modelo completo sirve.

## R²
Proporción de variabilidad explicada.

## AIC / BIC
Sirven para comparar modelos (menor = mejor).

no basta con mirar números sueltos.
Debemos hacer un análisis estructurado del modelo.



## 🧠 F-statistic: ¿El modelo sirve?

El *F-statistic* evalúa si el modelo completo realmente explica el resultado.

👉 Compara:
- Un modelo sin variables (solo el promedio)
- vs un modelo con variables (edad, días, severidad, etc.)

El valor 77.23 indica qué tan grande es la diferencia entre:

** modelo con variables vs modelo sin variables**

👉 Mientras más grande, más probable que el modelo sea útil

1. **F-statistic (77.23)**

👉 Es alto → sugiere que el modelo podría ser bueno

2. **Prob (F-statistic)**

👉 Esto confirma si es significativo

### 📊 ¿Cómo se interpreta?

Mira el valor:

**Prob (F-statistic)**

- ✔️ Si p < 0.05 → el modelo **sí sirve**
- ❌ Si p ≥ 0.05 → el modelo **no sirve**

### 🏥 En este contexto

Si p < 0.05:

👉 Las variables como días de estadía, severidad o servicio clínico ayudan a explicar el costo hospitalario.

---

💡 **Idea clave:**  
El F-statistic no dice qué variable es importante,  
solo dice si el modelo en conjunto tiene sentido.

# 5. Evaluación predictiva

Hasta ahora usamos el modelo para explicar relaciones (ej: cómo influye la severidad en el costo).
Pero también podemos usarlo para predecir resultados en pacientes nuevos.

👉 **¿Qué tan bien predice el modelo datos que no ha visto antes?**

Paso clave: separar los datos

Dividimos el dataset en dos partes:

Train (entrenamiento) → el modelo aprende
Test (prueba) → el modelo se evalúa

👉 Esto evita “hacer trampa” evaluando con los mismos datos usados para entrenar.

🧠 ¿Qué significa evaluar predictivamente?

Significa responder:

👉 ¿Qué tan bien predice el modelo datos que no ha visto antes?

🔀 Paso clave: separar los datos

Dividimos el dataset en dos partes:

Train (entrenamiento) → el modelo aprende
Test (prueba) → el modelo se evalúa

👉 Esto evita “hacer trampa” evaluando con los mismos datos usados para entrenar.

### Ejemplo en hospitalizaciones

Entrenamos el modelo con algunos pacientes, y luego lo usamos para predecir:

👉 costo hospitalario en pacientes nuevos

### Métricas de evaluación
🔹 **MAE (Mean Absolute Error)**: Error promedio del modelo

Ejemplo: MAE = 500.000

👉 El modelo se equivoca en promedio en $500.000 por paciente

🔹 **RMSE (Root Mean Squared Error)**

👉 Similar al MAE, pero penaliza más los errores grandes

Útil cuando errores grandes son especialmente problemáticos (ej: costos muy altos mal predichos)

🔹 **R² en test**

👉 Qué proporción de la variabilidad explica el modelo en datos nuevos

R² alto → buen poder predictivo
R² bajo → el modelo no predice bien
📉 Visualización clave

Gráfico:

👉 valor real vs valor predicho

Si los puntos están cerca de la diagonal → buen modelo
Si están dispersos → mala predicción

### Interpretación en salud

Un buen modelo predictivo permite:

anticipar costos hospitalarios
identificar pacientes de alto consumo de recursos
planificar camas y recursos clínicos
apoyar decisiones de gestión

In [4]:
train, test = train_test_split(df, test_size=0.25, random_state=42)

modelo_train = smf.ols(
    "costo ~ edad + dias_estadia + num_diag + num_proc + C(sexo) + C(servicio) + C(egreso) + C(severidad) + C(grd)",
    data=train
).fit()

pred = modelo_train.predict(test)

print("MAE:", mean_absolute_error(test["costo"], pred))
print("RMSE:", np.sqrt(mean_squared_error(test["costo"], pred)))
print("R2:", r2_score(test["costo"], pred))

MAE: 600404.9504274544
RMSE: 721309.7413296959
R2: 0.7728540528464729


## Interpretación

### 1. MAE (Error promedio)**

👉 MAE ≈ 600.000

Interpretación:

El modelo se equivoca en promedio en $600.000 por paciente

**En contexto hospitalario**

👉 Si estás prediciendo costo:

El error típico del modelo es de ~$600 mil
Es el “error esperado” al usar el modelo en pacientes nuevos

**Cómo evaluarlo**

Depende del rango del costo:

Si los costos son de $1–2 millones → error alto
Si son de $5–10 millones → error razonable


### 2. RMSE (Error penalizando errores grandes)

👉 RMSE ≈ 721.000

Interpretación:

El modelo tiene errores grandes de hasta ~$700.000 o más

🧠 Diferencia con MAE
MAE → error promedio
RMSE → castiga más errores grandes

👉 Como RMSE > MAE:

Hay algunos pacientes donde el modelo se equivoca bastante más

🏥 Insight clínico

👉 Probablemente:

casos complejos (UCI, alta severidad)
pacientes atípicos

### 3. R² (qué tanto explica el modelo)

👉 R² = 0.77

Interpretación:

El modelo explica el 77% de la variabilidad del costo hospitalario

Traducción simple: 👉 Es un modelo bastante bueno

**En salud**
R² > 0.7 → muy buen modelo
0.4–0.7 → aceptable
<0.3 → débil

👉 0.77 es alto para datos clínicos

### Interpretación global


El modelo presenta un buen desempeño predictivo, explicando aproximadamente el **77% de la variabilidad del costo** hospitalario. El error promedio de predicción es cercano a $600.000 por paciente, lo que sugiere una precisión razonable considerando la variabilidad clínica. Sin embargo, el RMSE indica que existen algunos casos con errores mayores, probablemente asociados a pacientes más complejos o con características no capturadas por el modelo.

Por lo tanto:
**El modelo explica bien el costo, pero aún se equivoca en algunos pacientes complejos.**

Buen R² → modelo explica bien
Error no trivial → aún faltan variables

# 6. Tarea final (GRD)

Construir un modelo con sus datos GRD.

Incluir:
- Variable dependiente
- Variables: edad, sexo, servicio, egreso, severidad, GRD
- Modelo simple y múltiple
- Interpretación clínica
- Evaluación
